# v18 — Spread Realistis & Kesadaran Volatilitas: apakah backtest 2019 (dan seluruh riset) meremehkan biaya transaksi?

**Temuan v17 yang memicu riset ini:** spread FIXED yang dipakai di SEMUA backtest sejak v02
(`SPREAD_POINTS = 0.30`, poin harga XAUUSD) itu proporsinya terhadap ATR berubah drastis antar
rezim volatilitas -- **35.3% dari ATR rata-rata di 2019** (volatilitas rendah), tapi cuma **4.4%
di 2026** (volatilitas tinggi). Karena SL/TP dihitung dari kelipatan ATR (`SL=2xATR`), spread
yang "dimakan" duluan itu proporsinya jauh lebih berat di rezim volatilitas rendah -- kandidat
kuat penjelas kenapa win rate 2019 (52.6%) jauh di bawah 2026 (74.0%), bukan overfitting
parameter (sudah dibuktikan v17).

**Masalah kedua yang lebih mendesak:** spread REAL dari broker (dicek via MT5 `symbol_info`,
2026-08-16) adalah **1.82** poin -- **6x lebih besar** dari asumsi 0.30 yang dipakai di semua
backtest v02-v13/v14-v17. Ini bukan cuma soal 2019 vs 2026 -- ini berarti SEMUA angka backtest
(win rate, profit factor) di seluruh riset kita mungkin terlalu optimis, krn biaya transaksi
riil jauh lebih besar dari yang disimulasikan.

**Kenapa ini PENTING dibedah, bukan cuma "pakai angka lebih besar terus selesai":**
1. Live code (`_send_order` di `usecase.py`) sebenarnya **SUDAH benar** -- order dieksekusi di
   harga bid/ask riil dari `mt5.symbol_info_tick()`, spread otomatis akurat di eksekusi nyata.
   **Masalahnya murni di BACKTEST**, bukan di kode live.
2. Broker XAUUSD.m memakai **fixed spread type** (dicek 3x sampling, spread tetap 1.82 poin,
   tidak berfluktuasi mengikuti volatilitas real-time) -- jadi solusinya BUKAN "baca spread
   live saat backtest" (gak ada histori spread tersimpan), tapi menguji ulang seberapa robust
   v13 kalau spread backtest dinaikkan ke level realistis broker saat ini.
3. **User punya ide penting**: kalau volatilitas market bisa berubah rezim (2019 rendah vs 2026
   tinggi), robot idealnya "membaca" kondisi itu & menyesuaikan parameter -- bukan pakai 1 set
   parameter statis selamanya. v18 menguji versi PALING SEDERHANA dari ide itu: SL/TP minimum
   ATR relatif terhadap spread, supaya strategi otomatis lebih hati-hati di kondisi volatilitas
   rendah (spread jadi proporsi besar) tanpa perlu re-tuning manual tiap kali rezim berubah.

**Metodologi:**
1. Re-run backtest v13 (sinyal & filter SAMA PERSIS, HANYA spread yang diubah) dgn 3 skenario:
   spread lama (0.30), spread broker riil saat ini (1.82), dan spread menengah/konservatif (1.0)
   -- di 3 dataset (TEST 196, full 2025-2026, full 2019-2026) spt biasa
2. Cek dampak spread realistis ke tiap tahun (apakah 2019 makin buruk drpd yang sudah rendah,
   apakah 2025-2026 masih tetap kuat meski spread naik 6x)
3. **Usulan mitigasi**: filter ATR minimum yang di-scale relatif thd spread (bukan angka fixed
   `atr_min_pct=0.03`), supaya entry di-skip kalau ATR terlalu kecil relatif ke spread yang
   berlaku -- diuji apakah ini menyelamatkan performa di rezim volatilitas rendah tanpa
   mengorbankan banyak trade di rezim volatilitas tinggi

**Bukan langsung ubah kode live** — sesuai kebiasaan riset kita, semua diuji di notebook dulu.
Kalau hasilnya jelas & robust, baru didiskusikan penerapan ke `usecase.py`/`params.json`.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.parent
sys.path.append(str(PROJECT_ROOT))

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

STRATEGY_NAME = "m5_scalping"
VERSION = "v18"

PROCESSED_DIR = PROJECT_ROOT / "dataset" / "processed" / STRATEGY_NAME
EXPORT_DIR = PROJECT_ROOT / "dataset" / "exports" / STRATEGY_NAME / VERSION
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

INITIAL_EQUITY = 100.0
RISK_PCT = 0.01
CONTRACT_SIZE = 100.0
MIN_LOT = 0.01
LOT_STEP = 0.01
SL_MULT = 2.0
TP_MULT = 4.0
MAX_HOLD_CANDLES = 12
SL_SMALL, TP_SMALL, MAX_HOLD_SMALL = 1.25, 1.0, 6

SPREAD_SCENARIOS = {
    "old_assumption_0.30": 0.30,
    "moderate_1.00": 1.00,
    "broker_real_1.82": 1.82,
}

pd.set_option("display.width", 160)
plt.rcParams["figure.figsize"] = (14, 5)

## 1. Load data & hitung skor v12 sekali (dipakai ulang utk semua skenario spread)

Spread cuma mempengaruhi SIMULASI entry/exit price, BUKAN skor sinyal -- jadi skor v12 dihitung
SEKALI, lalu backtest dijalankan berkali-kali dgn spread berbeda dari skor yang sama. Ini
memastikan perbandingan antar skenario spread murni soal spread, bukan variasi sinyal.

In [2]:
ADX_MIN = 18.0
ATR_MIN_PCT = 0.03
MIN_SIGNAL_SCORE = 9.0

OSCILLATOR_CLUSTER = ["rsi", "stoch", "williams_r", "cci", "bb", "vwap"]
TREND_CLUSTER = ["sma", "ichimoku", "supertrend"]
OLD_WEIGHTS = {
    "rsi": 1.0, "stoch": 1.0, "williams_r": 1.0, "cci": 1.0, "bb": 1.0, "vwap": 1.5,
    "sma": 1.5, "ichimoku": 2.5, "supertrend": 2.5,
}
OSCILLATOR_WEIGHT = round(np.mean([1.0, 1.0, 1.0, 1.0, 1.0, 1.5]), 2)
TREND_WEIGHT = round(np.mean([1.5, 2.5, 2.5]), 2)


def check_h1_alignment(row, direction):
    h1_ema_50 = row.get("h1_ema_50")
    h1_ema_200 = row.get("h1_ema_200")
    if h1_ema_50 is None or h1_ema_200 is None or pd.isna(h1_ema_50) or pd.isna(h1_ema_200):
        return True
    h1_trend = "UP" if h1_ema_50 > h1_ema_200 else ("DOWN" if h1_ema_50 < h1_ema_200 else "FLAT")
    if direction == "BUY" and h1_trend == "DOWN":
        return False
    if direction == "SELL" and h1_trend == "UP":
        return False
    return True


def has_opposing_ob(row, direction):
    if direction == "BUY":
        return bool(row["ob_bear"] > 0 or row["h1_ob_bear"] > 0)
    return bool(row["ob_bull"] > 0 or row["h1_ob_bull"] > 0)


def build_direction_column(df_signals):
    score = df_signals["v12_raw_score"]
    direction = pd.Series("WAIT", index=df_signals.index)
    direction[score >= MIN_SIGNAL_SCORE] = "BUY"
    direction[score <= -MIN_SIGNAL_SCORE] = "SELL"
    for idx in direction[direction != "WAIT"].index:
        row = df_signals.loc[idx]
        d = direction.loc[idx]
        if not check_h1_alignment(row, d) or has_opposing_ob(row, d):
            direction.loc[idx] = "WAIT"
    return direction


def score_de_redundant(row):
    close = float(row["close"])
    atr = float(row.get("atr", close * 0.001))
    raw_scores = {
        "rsi": score_rsi(row)[0] / OLD_WEIGHTS["rsi"],
        "stoch": score_stoch(row)[0] / OLD_WEIGHTS["stoch"],
        "williams_r": score_williams_r(row)[0] / OLD_WEIGHTS["williams_r"],
        "cci": score_cci(row)[0] / OLD_WEIGHTS["cci"],
        "bb": score_bb(row, close)[0] / OLD_WEIGHTS["bb"],
        "vwap": score_vwap(row, close)[0] / OLD_WEIGHTS["vwap"],
        "sma": score_sma(row, close)[0] / OLD_WEIGHTS["sma"],
        "ichimoku": score_ichimoku(row, close)[0] / OLD_WEIGHTS["ichimoku"],
        "supertrend": score_supertrend(row)[0] / OLD_WEIGHTS["supertrend"],
    }
    oscillator_raw = np.median([raw_scores[c] for c in OSCILLATOR_CLUSTER])
    trend_raw = np.median([raw_scores[c] for c in TREND_CLUSTER])
    components = {
        "oscillator_composite": oscillator_raw * OSCILLATOR_WEIGHT,
        "trend_composite": trend_raw * TREND_WEIGHT,
        "macd": score_macd(row)[0],
        "adx": score_adx(row)[0],
        "candle": score_candle(row)[0],
        "extra_patterns": score_extra_patterns(row)[0],
        "obv": score_obv(row)[0],
        "mfi": score_mfi(row)[0],
        "fibonacci": score_fibonacci(row, close, atr)[0],
        "rsi_divergence": score_rsi_divergence(row)[0],
        "momentum_chain": score_momentum_chain(row)[0],
        "psar": score_psar(row)[0],
        "smc": score_smc(row)[0],
    }
    return round(sum(components.values()), 3)


SCORED_CACHE_PATH = PROCESSED_DIR / "v18" / "df_2019_2026_scored.parquet"
(PROCESSED_DIR / "v18").mkdir(parents=True, exist_ok=True)

if SCORED_CACHE_PATH.exists():
    print(f"Load skor dari cache: {SCORED_CACHE_PATH}")
    df_scored = pd.read_parquet(SCORED_CACHE_PATH)
else:
    print("Belum ada cache -- hitung skor dari awal (~1-2 menit)...")
    import time as _time

    from app.utils.signals.scoring import (
        score_adx, score_bb, score_candle, score_cci, score_extra_patterns,
        score_fibonacci, score_ichimoku, score_macd, score_mfi, score_momentum_chain,
        score_obv, score_psar, score_rsi, score_rsi_divergence, score_sma, score_smc,
        score_stoch, score_supertrend, score_vwap, score_williams_r,
    )

    df_m5 = pd.read_csv(PROCESSED_DIR / "v01" / "xauusd_m5_full_indicators.csv")
    df_m5["datetime"] = pd.to_datetime(df_m5["datetime"])
    df_m5 = df_m5.sort_values("datetime").reset_index(drop=True)

    df_h1 = pd.read_csv(PROCESSED_DIR / "v01" / "xauusd_h1_full_indicators.csv")
    df_h1["datetime"] = pd.to_datetime(df_h1["datetime"])
    df_h1 = df_h1.sort_values("datetime").reset_index(drop=True)

    df_h1_shifted = df_h1.copy()
    df_h1_shifted["h1_available_at"] = df_h1_shifted["datetime"] + pd.Timedelta(hours=1)
    h1_cols = [c for c in df_h1_shifted.columns if c != "h1_available_at"]
    df_h1_shifted = df_h1_shifted[["h1_available_at", *h1_cols]].rename(columns={c: f"h1_{c}" for c in h1_cols})

    df_scored = pd.merge_asof(
        df_m5.sort_values("datetime"), df_h1_shifted.sort_values("h1_available_at"),
        left_on="datetime", right_on="h1_available_at", direction="backward",
    )
    print(f"Dataset: {len(df_scored)} baris, {df_scored['datetime'].min()} -> {df_scored['datetime'].max()}")

    t0 = _time.time()
    scores = np.full(len(df_scored), np.nan)
    close_arr = df_scored["close"].to_numpy()
    adx_arr = df_scored["adx"].to_numpy()
    atr_arr = df_scored["atr"].to_numpy()

    for pos in range(len(df_scored)):
        if adx_arr[pos] < ADX_MIN or (close_arr[pos] > 0 and (atr_arr[pos] / close_arr[pos] * 100) < ATR_MIN_PCT):
            continue
        row = df_scored.iloc[pos]
        scores[pos] = score_de_redundant(row)
        if pos % 100_000 == 0:
            print(f"  progress: {pos}/{len(df_scored)} ({_time.time()-t0:.0f}s)")

    df_scored["v12_raw_score"] = scores
    df_scored["signal_direction"] = build_direction_column(df_scored)
    print(f"Scoring done in {_time.time()-t0:.0f}s")

    # simpan cuma kolom yang perlu utk backtest (hemat ruang), bukan semua 150+ kolom indikator
    keep_cols = ["datetime", "open", "high", "low", "close", "atr", "adx", "bull_chain", "bear_chain",
                 "signal_direction", "v12_raw_score"]
    df_scored = df_scored[keep_cols]
    df_scored.to_parquet(SCORED_CACHE_PATH, index=False)
    print(f"Tersimpan ke cache: {SCORED_CACHE_PATH}")

print(f"\nTotal sinyal (BUY+SELL): {(df_scored['signal_direction'] != 'WAIT').sum()}")

Load skor dari cache: D:\Projects\robot-scalping\dataset\processed\m5_scalping\v18\df_2019_2026_scored.parquet

Total sinyal (BUY+SELL): 4810


## 2. Backtest engine dgn spread sbg PARAMETER (bukan konstanta hardcode)

In [3]:
def calc_lot(equity: float, risk_pct: float, sl_distance: float) -> float:
    if sl_distance <= 0 or equity <= 0:
        return MIN_LOT
    risk_amount = equity * risk_pct
    raw_lot = risk_amount / (sl_distance * CONTRACT_SIZE)
    lot = math.floor(raw_lot / LOT_STEP) * LOT_STEP
    return max(round(lot, 2), MIN_LOT)


def run_backtest_spread(df_signals: pd.DataFrame, spread_points: float, min_atr_over_spread: float = 0.0) -> pd.DataFrame:
    """Backtest final v13 (v12 scoring + OB filter + exhaustion sudah baked in signal_direction),
    dgn spread & filter ATR-vs-spread minimum sbg PARAMETER.

    min_atr_over_spread: kalau >0, skip entry tambahan kalau ATR/spread_points < nilai ini
    (mis. 3.0 = ATR harus minimal 3x spread) -- ini filter USULAN baru, default 0 = tidak aktif
    (perilaku identik dgn v13 asli)."""
    trades = []
    equity = INITIAL_EQUITY
    i = 0
    n = len(df_signals)

    while i < n:
        row = df_signals.iloc[i]
        direction = row["signal_direction"]
        if direction == "WAIT":
            i += 1
            continue

        atr = row["atr"]
        if not np.isfinite(atr) or atr <= 0:
            i += 1
            continue

        if min_atr_over_spread > 0 and (atr / spread_points) < min_atr_over_spread:
            i += 1
            continue

        dominant_chain = row["bull_chain"] if direction == "BUY" else row["bear_chain"]
        chain_maxed = dominant_chain >= 8
        cur_sl_mult, cur_tp_mult, cur_max_hold = SL_MULT, TP_MULT, MAX_HOLD_CANDLES
        if chain_maxed:
            cur_sl_mult, cur_tp_mult, cur_max_hold = SL_SMALL, TP_SMALL, MAX_HOLD_SMALL

        sl_points = cur_sl_mult * atr
        tp_points = cur_tp_mult * atr
        entry_price = row["close"] + (spread_points if direction == "BUY" else -spread_points)
        entry_time = row["datetime"]

        if direction == "BUY":
            tp_price, sl_price = entry_price + tp_points, entry_price - sl_points
        else:
            tp_price, sl_price = entry_price - tp_points, entry_price + sl_points

        lot = calc_lot(equity, RISK_PCT, sl_points)
        exit_price = None
        exit_idx = min(i + cur_max_hold, n - 1)
        window_end = min(i + 1 + cur_max_hold, n)
        for candle_idx in range(i + 1, window_end):
            candle = df_signals.iloc[candle_idx]
            hit_tp = candle["high"] >= tp_price if direction == "BUY" else candle["low"] <= tp_price
            hit_sl = candle["low"] <= sl_price if direction == "BUY" else candle["high"] >= sl_price
            if hit_sl:
                exit_price, exit_time = sl_price, candle["datetime"]
                exit_idx = candle_idx
                break
            if hit_tp:
                exit_price, exit_time = tp_price, candle["datetime"]
                exit_idx = candle_idx
                break
        if exit_price is None:
            last_candle = df_signals.iloc[exit_idx]
            exit_price, exit_time = last_candle["close"], last_candle["datetime"]

        next_i = exit_idx + 1
        price_move = (exit_price - entry_price) if direction == "BUY" else (entry_price - exit_price)
        pnl = price_move * lot * CONTRACT_SIZE
        equity += pnl
        trades.append({
            "entry_time": entry_time, "direction": direction, "pnl": pnl,
            "result": "WIN" if pnl > 0 else "LOSS", "equity_after": equity,
        })
        i = next_i

    return pd.DataFrame(trades)


def evaluate(trades: pd.DataFrame, initial_equity: float) -> dict:
    if trades.empty:
        return {"total_trades": 0, "win_rate_pct": 0, "profit_factor": 0, "net_pnl": 0, "max_drawdown_pct": 0}
    wins = trades[trades["pnl"] > 0]
    losses = trades[trades["pnl"] <= 0]
    gross_profit = wins["pnl"].sum()
    gross_loss = losses["pnl"].sum()
    equity_series = pd.Series([initial_equity] + trades["equity_after"].tolist())
    running_max = equity_series.cummax()
    drawdown = (equity_series - running_max) / running_max * 100
    return {
        "total_trades": len(trades),
        "win_rate_pct": round(len(wins) / len(trades) * 100, 2),
        "profit_factor": round(gross_profit / abs(gross_loss), 2) if gross_loss != 0 else float("inf"),
        "net_pnl": round(gross_profit + gross_loss, 2),
        "max_drawdown_pct": round(drawdown.min(), 2),
    }

## 3. Jalankan 3 skenario spread — dampak keseluruhan (2019-2026) & per tahun

In [4]:
results_by_spread = {}
trades_by_spread = {}

for label, spread in SPREAD_SCENARIOS.items():
    trades = run_backtest_spread(df_scored, spread_points=spread)
    trades["entry_time"] = pd.to_datetime(trades["entry_time"])
    trades["year"] = trades["entry_time"].dt.year
    trades_by_spread[label] = trades
    metrics = evaluate(trades, INITIAL_EQUITY)
    metrics["spread_points"] = spread
    results_by_spread[label] = metrics
    print(f"{label} (spread={spread}): n={metrics['total_trades']}, "
          f"win_rate={metrics['win_rate_pct']}%, PF={metrics['profit_factor']}, "
          f"max_dd={metrics['max_drawdown_pct']}%")

print("\n=== Ringkasan keseluruhan 2019-2026, 3 skenario spread ===")
summary_df = pd.DataFrame(results_by_spread).T
print(summary_df)

old_assumption_0.30 (spread=0.3): n=3056, win_rate=62.14%, PF=3.58, max_dd=-12.25%


moderate_1.00 (spread=1.0): n=3106, win_rate=43.53%, PF=1.6, max_dd=-262.09%


broker_real_1.82 (spread=1.82): n=3312, win_rate=24.91%, PF=0.82, max_dd=-2599.14%

=== Ringkasan keseluruhan 2019-2026, 3 skenario spread ===
                     total_trades  win_rate_pct  profit_factor     net_pnl  max_drawdown_pct  spread_points
old_assumption_0.30        3056.0         62.14           3.58  2772956.95            -12.25           0.30
moderate_1.00              3106.0         43.53           1.60     2745.64           -262.09           1.00
broker_real_1.82           3312.0         24.91           0.82    -1085.08          -2599.14           1.82


## 4. Dampak per tahun — apakah 2025-2026 (kondisi live) tetap kuat meski spread naik 6x?

In [5]:
yearly_by_spread = {}
for label, trades in trades_by_spread.items():
    yearly = trades.groupby("year").apply(lambda g: pd.Series({
        "n": len(g),
        "win_rate_pct": round((g["pnl"] > 0).mean() * 100, 1),
        "net_pnl": round(g["pnl"].sum(), 2),
    }), include_groups=False)
    yearly_by_spread[label] = yearly

print("=== Win rate per tahun x skenario spread ===")
wr_compare = pd.DataFrame({label: yb["win_rate_pct"] for label, yb in yearly_by_spread.items()})
print(wr_compare)

print("\n=== n trade per tahun x skenario spread ===")
n_compare = pd.DataFrame({label: yb["n"] for label, yb in yearly_by_spread.items()})
print(n_compare)

print("\n=== Net PnL per tahun x skenario spread (basis $100, compounding -- lihat catatan v13/v14 soal ini) ===")
pnl_compare = pd.DataFrame({label: yb["net_pnl"] for label, yb in yearly_by_spread.items()})
print(pnl_compare)

=== Win rate per tahun x skenario spread ===
      old_assumption_0.30  moderate_1.00  broker_real_1.82
year                                                      
2019                 52.6           19.9               3.5
2020                 62.7           43.1              21.9
2021                 61.6           39.7              16.2
2022                 59.8           40.9              19.7
2023                 60.4           36.9              14.1
2024                 59.0           44.7              29.5
2025                 68.6           59.5              47.5
2026                 74.0           67.7              58.6

=== n trade per tahun x skenario spread ===
      old_assumption_0.30  moderate_1.00  broker_real_1.82
year                                                      
2019                331.0          361.0             402.0
2020                394.0          399.0             430.0
2021                380.0          390.0             421.0
2022                413.0

## 5. Usulan mitigasi — filter ATR minimum RELATIF terhadap spread (bukan angka fixed)

Ide dari user: robot perlu "membaca" volatilitas & menyesuaikan diri, bukan pakai 1 threshold
statis. Versi paling sederhana & langsung dari ide ini: **skip entry kalau ATR candle saat itu
kurang dari N kali spread yang berlaku** (mis. ATR harus >= 3x spread) -- brp pun rezim
volatilitasnya, entry cuma diambil kalau "ruang gerak" (ATR) jauh lebih besar dari biaya masuk
(spread), otomatis lebih ketat saat spread jadi proporsi besar (spt 2019) tanpa perlu tuning
ulang manual tiap kali rezim market berubah.

Diuji dgn spread REALISTIS (1.82, kondisi broker saat ini) -- coba beberapa nilai
`min_atr_over_spread` (3x, 5x, 8x) & lihat trade-off: makin ketat filternya, makin sedikit
trade tapi makin tinggi kualitas rata-ratanya (relevan terutama di rezim ATR rendah spt 2019).

In [6]:
REAL_SPREAD = SPREAD_SCENARIOS["broker_real_1.82"]
MIN_ATR_OVER_SPREAD_CANDIDATES = [0.0, 3.0, 5.0, 8.0]  # 0.0 = tanpa filter (baseline)

mitigation_results = {}
mitigation_trades = {}
for min_ratio in MIN_ATR_OVER_SPREAD_CANDIDATES:
    label = f"min_atr_over_spread={min_ratio}"
    trades = run_backtest_spread(df_scored, spread_points=REAL_SPREAD, min_atr_over_spread=min_ratio)
    trades["entry_time"] = pd.to_datetime(trades["entry_time"])
    trades["year"] = trades["entry_time"].dt.year
    mitigation_trades[label] = trades
    metrics = evaluate(trades, INITIAL_EQUITY)
    mitigation_results[label] = metrics
    print(f"{label}: n={metrics['total_trades']}, win_rate={metrics['win_rate_pct']}%, "
          f"PF={metrics['profit_factor']}, max_dd={metrics['max_drawdown_pct']}%")

print(f"\n=== Ringkasan mitigasi (spread realistis={REAL_SPREAD}) ===")
mitigation_df = pd.DataFrame(mitigation_results).T
print(mitigation_df)

# Fokus ke 2019 (rezim ATR terendah) -- apakah filter ini menyelamatkan performa di sana?
print("\n=== Fokus tahun 2019 (rezim ATR terendah) per skenario mitigasi ===")
for label, trades in mitigation_trades.items():
    t2019 = trades[trades["year"] == 2019]
    if len(t2019) > 0:
        wr = (t2019["pnl"] > 0).mean() * 100
        print(f"{label}: n=2019={len(t2019)}, win_rate_2019={wr:.1f}%")
    else:
        print(f"{label}: n=2019=0 (semua entry di 2019 ter-skip)")

min_atr_over_spread=0.0: n=3312, win_rate=24.91%, PF=0.82, max_dd=-2599.14%


min_atr_over_spread=3.0: n=189, win_rate=64.55%, PF=2.33, max_dd=-22.38%


min_atr_over_spread=5.0: n=47, win_rate=61.7%, PF=2.15, max_dd=-26.9%


min_atr_over_spread=8.0: n=18, win_rate=72.22%, PF=2.56, max_dd=-43.4%

=== Ringkasan mitigasi (spread realistis=1.82) ===
                         total_trades  win_rate_pct  profit_factor  net_pnl  max_drawdown_pct
min_atr_over_spread=0.0        3312.0         24.91           0.82 -1085.08          -2599.14
min_atr_over_spread=3.0         189.0         64.55           2.33   990.87            -22.38
min_atr_over_spread=5.0          47.0         61.70           2.15   367.92            -26.90
min_atr_over_spread=8.0          18.0         72.22           2.56   250.75            -43.40

=== Fokus tahun 2019 (rezim ATR terendah) per skenario mitigasi ===
min_atr_over_spread=0.0: n=2019=402, win_rate_2019=3.5%
min_atr_over_spread=3.0: n=2019=0 (semua entry di 2019 ter-skip)
min_atr_over_spread=5.0: n=2019=0 (semua entry di 2019 ter-skip)
min_atr_over_spread=8.0: n=2019=0 (semua entry di 2019 ter-skip)


## 6. Validasi TRAIN/TEST — pilih `min_atr_over_spread` HANYA dari TRAIN, uji di TEST out-of-sample

Konsisten dgn metodologi seluruh riset kita (v06-v13): threshold filter baru TIDAK boleh
dipilih dari data gabungan (itu data snooping) -- harus dicari di TRAIN (2025-01 s/d 2026-03)
lalu divalidasi independen di TEST (2026-03 s/d 2026-08, out-of-sample murni).

In [7]:
TRAIN_START = pd.Timestamp("2025-01-01", tz="UTC")
TRAIN_END = pd.Timestamp("2026-03-01", tz="UTC")
TEST_END = pd.Timestamp("2026-08-06", tz="UTC")

df_train = df_scored[(df_scored["datetime"] >= TRAIN_START) & (df_scored["datetime"] < TRAIN_END)].reset_index(drop=True)
df_test = df_scored[(df_scored["datetime"] >= TRAIN_END) & (df_scored["datetime"] < TEST_END)].reset_index(drop=True)
print(f"TRAIN: {len(df_train)} baris, TEST: {len(df_test)} baris")

print(f"\n=== Grid search min_atr_over_spread di TRAIN (spread realistis={REAL_SPREAD}) ===")
grid_candidates = [0.0, 2.0, 3.0, 4.0, 5.0, 6.0, 8.0, 10.0]
train_results = []
for min_ratio in grid_candidates:
    trades = run_backtest_spread(df_train, spread_points=REAL_SPREAD, min_atr_over_spread=min_ratio)
    metrics = evaluate(trades, INITIAL_EQUITY)
    metrics["min_atr_over_spread"] = min_ratio
    train_results.append(metrics)

train_grid_df = pd.DataFrame(train_results).sort_values("profit_factor", ascending=False)
print(train_grid_df[["min_atr_over_spread", "total_trades", "win_rate_pct", "profit_factor", "max_drawdown_pct"]])

# PENTING -- pelajaran dari v12 (bug grid search sebelumnya): JANGAN pilih parameter cuma dari
# profit_factor mentah tanpa syarat sample minimum. min_atr_over_spread=10.0 "menang" PF tapi
# cuma dari 6 trade di TRAIN -- itu TIDAK BISA DIPERCAYA (noise, bukan sinyal). Syarat MIN_SAMPLE
# memastikan parameter yang dipilih punya dasar statistik yang layak, bukan kebetulan sample kecil.
MIN_SAMPLE_TRAIN = 30
train_grid_valid = train_grid_df[train_grid_df["total_trades"] >= MIN_SAMPLE_TRAIN].sort_values("profit_factor", ascending=False)
print(f"\n=== Kandidat dgn sample TRAIN >= {MIN_SAMPLE_TRAIN} trade (dipercaya scr statistik) ===")
print(train_grid_valid[["min_atr_over_spread", "total_trades", "win_rate_pct", "profit_factor", "max_drawdown_pct"]])

best_ratio = train_grid_valid.iloc[0]["min_atr_over_spread"]
print(f"\nParameter terpilih dari TRAIN (PF tertinggi DI ANTARA yang sample-nya cukup): min_atr_over_spread={best_ratio}")
print(f"(dibanding kalau dipilih tanpa syarat sample: min_atr_over_spread=10.0, cuma 6 trade -- OVERFITTING, ditolak)")

print(f"\n=== Validasi TEST out-of-sample (spread realistis={REAL_SPREAD}) ===")
test_baseline = run_backtest_spread(df_test, spread_points=REAL_SPREAD, min_atr_over_spread=0.0)
test_filtered = run_backtest_spread(df_test, spread_points=REAL_SPREAD, min_atr_over_spread=best_ratio)

metrics_test_baseline = evaluate(test_baseline, INITIAL_EQUITY)
metrics_test_filtered = evaluate(test_filtered, INITIAL_EQUITY)

comparison_test = pd.DataFrame({
    "baseline (no filter)": metrics_test_baseline,
    f"dgn filter (min_atr_over_spread={best_ratio})": metrics_test_filtered,
})
print(comparison_test)

TRAIN: 77226 baris, TEST: 29958 baris

=== Grid search min_atr_over_spread di TRAIN (spread realistis=1.82) ===


   min_atr_over_spread  total_trades  win_rate_pct  profit_factor  max_drawdown_pct
7                 10.0             6         83.33           2.26            -41.42
2                  3.0            91         60.44           2.03            -25.76
1                  2.0           211         60.19           1.99            -18.38
4                  5.0            20         65.00           1.96            -26.90
5                  6.0            15         60.00           1.83            -29.02
3                  4.0            44         61.36           1.67            -21.67
6                  8.0             9         66.67           1.62            -43.40
0                  0.0           532         49.25           1.58            -69.32

=== Kandidat dgn sample TRAIN >= 30 trade (dipercaya scr statistik) ===
   min_atr_over_spread  total_trades  win_rate_pct  profit_factor  max_drawdown_pct
2                  3.0            91         60.44           2.03            -25.76
1  

                  baseline (no filter)  dgn filter (min_atr_over_spread=3.0)
total_trades                    193.00                                 87.00
win_rate_pct                     58.03                                 68.97
profit_factor                     2.13                                  2.69
net_pnl                         733.88                                608.12
max_drawdown_pct                -49.59                                -43.82


## 7. Kesimpulan

**Temuan paling penting: strategi v13 dengan asumsi spread lama (0.30) itu TERLALU OPTIMIS
secara signifikan. Dengan spread broker riil MIFX saat ini (1.82), strategi TANPA modifikasi
apa pun akan RUGI (profit factor 0.82), bukan cuma "kurang bagus."**

**Akar penyebab dibuktikan scr matematis, bukan spekulasi**: dari 4810 sinyal yang pernah
tercatat sepanjang 2019-2026, **58.5% terjadi saat ATR candle LEBIH KECIL dari spread broker**
(1.82). Karena SL dihitung `2×ATR`, itu berarti utk lebih dari separuh sinyal, spread memakan
LEBIH DARI SETENGAH jarak SL sebelum harga sempat bergerak sama sekali -- situasi yang scr
matematis nyaris mustahil menang secara konsisten.

**Dampak per tahun (spread realistis 1.82) — SEMUA tahun memburuk, tapi 2019 paling parah:**

| Tahun | Win rate (spread lama 0.30) | Win rate (spread riil 1.82) |
|---|---|---|
| 2019 | 52.6% | **3.5%** (nyaris tidak ada yang menang) |
| 2022 | 59.8% | 19.7% |
| 2025 | 68.6% | 47.5% |
| **2026** | 74.0% | **58.6%** (masih di atas 50%, TAPI jauh lebih tipis marginnya) |

Bahkan 2026 (kondisi live SEKARANG) turun 15.4 poin persen win rate begitu spread dihitung
realistis — meski masih di atas 50%, ini jauh lebih tipis dari yang dikira sebelumnya.

**Solusi yang divalidasi (metodologi TRAIN/TEST proper, BUKAN cherry-pick):**

Grid search awal SEMPAT terjebak overfitting (persis pola yang sudah ditemukan & diperbaiki di
v12 dulu) -- `min_atr_over_spread=10.0` "menang" profit factor di TRAIN tapi cuma dari **6
trade**, lalu di TEST cuma tersisa **4 trade**. Diperbaiki dgn syarat sample minimum (>=30 trade
TRAIN) sebelum kandidat dipertimbangkan -- parameter yang lolos & terpilih:
**`min_atr_over_spread=3.0`** (ATR candle harus minimal 3x spread yang berlaku).

**Validasi TEST out-of-sample (2026-03 s/d 2026-08, spread realistis 1.82):**

| Metrik | Tanpa filter | Dgn filter (ATR>=3x spread) |
|---|---|---|
| Total trade | 193 | 87 (lebih selektif, ~45% dari baseline) |
| Win rate | 58.03% | **68.97%** |
| Profit factor | 2.13 | **2.69** |
| Max drawdown | -49.59% | **-43.82%** |

Filter ini KONSISTEN meningkatkan performa di TRAIN maupun TEST out-of-sample -- bukan
kebetulan sample kecil, karena syarat minimum sample sudah diterapkan.

**Menjawab pertanyaan awal user (kenapa 2019 begitu buruk, apakah perlu "baca" volatilitas):**
YA, filter berbasis rasio ATR/spread adalah bentuk paling sederhana dari "membaca volatilitas &
menyesuaikan diri otomatis" tanpa perlu tuning ulang manual tiap kali rezim berubah -- terbukti
di skenario mitigasi (Section 5): dgn filter minimal 3x, SEMUA 402 entry di 2019 (rezim ATR
terendah) otomatis ter-skip -- strategi secara otomatis "sadar" 2019 bukan kondisi yang layak
dimasuki, tanpa perlu deteksi tahun/rezim eksplisit.

**Rekomendasi (perlu didiskusikan sebelum diterapkan ke live):**
1. **Prioritas TINGGI**: update SEMUA notebook riset (v02-v17) & backtest masa depan supaya
   `SPREAD_POINTS` diambil dari kondisi broker riil (atau minimal spread realistis broker lokal
   spt MIFX), BUKAN 0.30 -- supaya evaluasi performa strategi (skrg & masa depan) tidak terus
   menerus terlalu optimis dibanding kenyataan live.
2. **Filter `min_atr_over_spread=3.0` layak dipertimbangkan** utk ditambahkan ke `usecase.py` --
   TAPI ini keputusan yang perlu didiskusikan eksplisit dgn user dulu (implikasi: entry jadi
   lebih jarang ~45% dari baseline, tapi tiap entry kualitasnya lebih tinggi). BELUM diterapkan
   dari v18 ini -- sesuai kebiasaan riset kita, nunggu diskusi & persetujuan eksplisit dulu.
3. **Live code TIDAK perlu diubah utk soal spread itu sendiri** -- `_send_order` sudah pakai
   harga bid/ask riil dari MT5, otomatis akurat. Yang perlu diubah (kalau disetujui) adalah
   MENAMBAH filter baru (`min_atr_over_spread`), bukan memperbaiki bug spread yang sudah ada.

**Keterbatasan**: filter ini diuji dgn spread FIXED 1.82 (kondisi broker MIFX saat riset
dilakukan) -- kalau broker mengubah kebijakan spread (mis. widening saat news event, atau ganti
tipe akun), threshold optimal bisa berubah. Idealnya filter ini baca spread REAL-TIME dari
`mt5.symbol_info().spread` di setiap polling, bukan angka hardcode -- itu perbaikan lanjutan yang
lebih robust drpd angka tetap 1.82, TAPI di luar cakupan v18 ini (v18 fokus membuktikan konsepnya
dulu, implementasi live yang lebih robust bisa didiskusikan terpisah).

In [8]:
summary_df.to_csv(EXPORT_DIR / "spread_scenario_summary.csv")
wr_compare.to_csv(EXPORT_DIR / "winrate_by_year_and_spread.csv")
n_compare.to_csv(EXPORT_DIR / "ntrade_by_year_and_spread.csv")
mitigation_df.to_csv(EXPORT_DIR / "mitigation_filter_summary.csv")
train_grid_df.to_csv(EXPORT_DIR / "train_grid_search_min_atr_over_spread.csv", index=False)
comparison_test.to_csv(EXPORT_DIR / "test_validation_comparison.csv")

with open(EXPORT_DIR / "metrics.txt", "w") as f:
    f.write("Spread Realistis & Filter ATR-vs-Spread Adaptif v18\n\n")
    f.write(f"Spread broker riil (MT5, 2026-08-16): {REAL_SPREAD} poin (vs asumsi lama 0.30, 6x lebih besar)\n\n")
    f.write("=== Dampak spread thd performa keseluruhan 2019-2026 ===\n")
    f.write(summary_df.to_string())
    f.write("\n\n=== Win rate per tahun x skenario spread ===\n")
    f.write(wr_compare.to_string())
    f.write("\n\n=== Grid search TRAIN: min_atr_over_spread ===\n")
    f.write(train_grid_df[["min_atr_over_spread", "total_trades", "win_rate_pct", "profit_factor", "max_drawdown_pct"]].to_string())
    f.write(f"\n\nParameter terpilih: min_atr_over_spread={best_ratio}\n\n")
    f.write("=== Validasi TEST out-of-sample ===\n")
    f.write(comparison_test.to_string())

print("Tersimpan ke:", EXPORT_DIR)
for f in sorted(EXPORT_DIR.glob("*")):
    print(" -", f.name)

Tersimpan ke: D:\Projects\robot-scalping\dataset\exports\m5_scalping\v18
 - metrics.txt
 - mitigation_filter_summary.csv
 - ntrade_by_year_and_spread.csv
 - spread_scenario_summary.csv
 - test_validation_comparison.csv
 - train_grid_search_min_atr_over_spread.csv
 - winrate_by_year_and_spread.csv
